# CelebA Flow Matching Demo

Generate 64x64 face samples from the published flow-matching model.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from huggingface_hub import snapshot_download

REPO_ID = "sunnycloudhust/CelebA-flow-matching"
MODEL_DIR = Path(snapshot_download(REPO_ID, allow_patterns=["config.json", "modeling.py", "pytorch_model.bin"]))
sys.path.insert(0, str(MODEL_DIR))

In [ ]:
from modeling import FlowMatchingModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FlowMatchingModel.from_pretrained(MODEL_DIR).to(device)
model.eval()
print(f"Running on {device}")

In [ ]:
@torch.no_grad()
def sample(model, n_samples=4, steps=50, image_size=64, device="cpu"):
    x = torch.randn(n_samples, 3, image_size, image_size, device=device)
    time_grid = torch.linspace(0, 1, steps + 1, device=device)
    for start, end in zip(time_grid[:-1], time_grid[1:]):
        t = torch.full((n_samples,), start, device=device)
        x = x + (end - start) * model(x, t)
    return x.clamp(-1, 1)

samples = sample(model, n_samples=4, steps=50, device=device)
samples = (samples.cpu() + 1) / 2

In [ ]:
fig, axes = plt.subplots(1, len(samples), figsize=(10, 3))
for axis, image in zip(axes, samples):
    axis.imshow(image.permute(1, 2, 0).numpy())
    axis.axis("off")
plt.tight_layout()
plt.show()